In [1]:
# Set working directory
import os
os.chdir("../../")

In [2]:
# Configure paths

# Promoter-level chec-seq binding signal
sumprom_chec_glob = "sumproms/*.gz"

# Motif binding score directory (see analysis code for calculation)
motif_binding_directory = "binding_score_at_motifs"

## Imports

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from itertools import combinations
import glob
import matplotlib.pyplot as plt

## Load data

In [4]:
sumprom_chec_files = glob.glob(sumprom_chec_glob)
sumprom_all = pd.concat([pd.read_parquet(x) for x in sumprom_chec_files], axis=1)

corr_cutoff = 0.895

def filter_reproducible(sumprom_all: pd.DataFrame, cutoff) -> pd.DataFrame:
    df = sumprom_all.copy()
    groups = pd.Series(df.columns, index=df.columns).str.rsplit("_", n=2).str[0]
    
    keep = []
    for _, members in groups.groupby(groups).groups.items():
        if len(members) < 2:
            continue
        corr = df[members].corr()
        np.fill_diagonal(corr.values, np.nan)
        max_corrs = corr.max(axis=1)
        reproducible = max_corrs[max_corrs >= cutoff].index.tolist()
        keep.extend(reproducible)
    return df[keep]

sumprom_filtered = filter_reproducible(sumprom_all, cutoff=corr_cutoff)

In [5]:
cols = pd.Index(sumprom_filtered.columns)
prefix = cols.to_series().str.rsplit("_", n=2).str[0]
best3 = lambda g: list(g) if len(g) <= 3 else max(combinations(g, 3), key=lambda c: sumprom_filtered.loc[:, c].corr().to_numpy()[np.triu_indices(3, 1)].sum())
sumprom_filtered = sumprom_filtered.loc[:, cols.to_series().groupby(prefix).apply(best3).explode()]
sumprom_all = sumprom_filtered.T.groupby(prefix).mean().T

FOXK1_WT = ["FOXP3","FOXA2","FOXF1","FOXL1","FOXL2","FOXJ2","FOXO3","FOXP1","FOXP2"]
GABPA_WT = ["ELF1","ELF2","ERF1","ELK1","ELK4","ERG","FLI1"]

dbd_fam_dict = {"FOXK1": FOXK1_WT, "GABPA": GABPA_WT}

cols_to_keep = set(sum(dbd_fam_dict.values(), []))
sumprom = sumprom_all.loc[:, sumprom_all.columns.isin(cols_to_keep)]
sumprom_z = sumprom.apply(zscore, axis=0)

In [6]:
corr = sumprom_z.corr()

In [7]:
def load_sample(sample_name, regex_search):
    if len(regex_search) == 1:
        df_files = glob.glob(f"{motif_binding_directory}/{sample_name}/{sample_name}{regex_search[0]}")
    if len(regex_search) == 2:
        df_files = glob.glob(f"{motif_binding_directory}/{sample_name}/{sample_name}{regex_search[0]}") + glob.glob(f"{motif_binding_directory}/{sample_name}/{sample_name}{regex_search[1]}")
    if len(df_files) != 1:
        return None, None
    bg_files = glob.glob(f"{motif_binding_directory}/background_binding_arrays/{sample_name}__*__loc-prom__signal_bg__fl25*.npy")
    if len(bg_files) != 1:
        return None, None
    df = pd.read_csv(df_files[0])
    return df

In [8]:
all_samples = set(sum(dbd_fam_dict.values(), []))

df_dict_representative = {}
for sample in all_samples:
    df_dict_representative[sample] = load_sample(sample, regex_search=["__*loc-prom__*family*__fl25__*.csv"])

## Helper functions

In [9]:
def get_sample_replicate_cols(sumprom_filtered, sample: str):
    # match up to the second-to-last underscore block: "ATF1_2_S5" -> "ATF1_2"
    return [c for c in sumprom_filtered.columns if "_".join(c.split("_")[:-2]) == sample]


def binding_corr_for_family(samples, df_dict_representative):
    # sample-level correlation of z_score_norm_f7 after exact alignment on (motif_id, sequence_name, start)
    wide = {}
    for s in samples:
        df = df_dict_representative.get(s, None)
        if not isinstance(df, pd.DataFrame):
            continue
        need = {"motif_id", "sequence_name", "start", "z_score_norm_f7"}
        if not need.issubset(df.columns):
            continue
        wide[s] = df.set_index(["motif_id", "sequence_name", "start"])["z_score_norm_f7"]

    if len(wide) < 2:
        return None

    wide_df = pd.concat(wide, axis=1)
    return wide_df.corr()  # pairwise-complete correlation

## Plot

In [10]:
def plot_families_unstitched(dbd_fam_dict, sumprom_filtered, df_dict_representative, label_fontsize=12):
    diag_lw = 1.1
    diag_color = "black"
    cmap = plt.get_cmap("coolwarm")

    def _is_valid_binding_df(df):
        # Check required columns exist
        need = {"motif_id", "sequence_name", "start", "z_score_norm_f7"}
        return isinstance(df, pd.DataFrame) and need.issubset(df.columns)

    def _cluster_order(samples, bind_corr):
        # Try hierarchical clustering to reorder samples
        try:
            from scipy.cluster.hierarchy import linkage, leaves_list
            from scipy.spatial.distance import squareform

            D = 1.0 - bind_corr.values
            np.fill_diagonal(D, 0.0)
            Z = linkage(squareform(D, checks=False), method="average")
            return [samples[i] for i in leaves_list(Z)]
        except Exception:
            return samples

    def _edges(n, scale, offset=0):
        # Compute integer pixel boundaries (avoids drift/misalignment)
        return [int(round(offset + k * scale / n)) for k in range(n + 1)]

    # Collect valid families
    fam_items = []
    for fam, samples in dbd_fam_dict.items():
        kept, reps = [], {}
        for s in samples:
            df = df_dict_representative.get(s, None)
            cols = sorted(get_sample_replicate_cols(sumprom_filtered, s))
            if _is_valid_binding_df(df) and len(cols) > 0:
                kept.append(s)
                reps[s] = cols
        if len(kept) >= 2:
            fam_items.append((fam, kept, reps))

    if not fam_items:
        return

    # Sort families by size, then name
    fam_items.sort(key=lambda x: (-len(x[1]), x[0]))

    scale = 100

    for fam, kept, reps in fam_items:
        n = len(kept)
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_axes([0.22, 0.12, 0.62, 0.76])

        # Build binding correlation matrix
        wide = {}
        for s in kept:
            df = df_dict_representative[s][["motif_id", "sequence_name", "start", "z_score_norm_f7"]]
            wide[s] = df.set_index(["motif_id", "sequence_name", "start"])["z_score_norm_f7"]

        bind_corr = pd.concat(wide, axis=1).corr()

        # Reorder samples
        kept = _cluster_order(kept, bind_corr)
        bind_corr = bind_corr.loc[kept, kept].clip(lower=0)

        # Compute replicate correlation
        all_rep_cols = [c for s in kept for c in reps[s]]
        rep_corr = sumprom_filtered[all_rep_cols].corr().clip(lower=0)

        # Create empty image
        img = np.full((n * scale, n * scale), np.nan)

        # Fill lower-left with binding correlations
        for i, si in enumerate(kept):
            for j, sj in enumerate(kept):
                if i > j:
                    img[i*scale:(i+1)*scale, j*scale:(j+1)*scale] = bind_corr.loc[si, sj]

        # Fill upper-right with replicate tiles
        for i, si in enumerate(kept):
            ri = reps[si]
            yi = _edges(len(ri), scale, i * scale)

            for j, sj in enumerate(kept):
                if i < j:
                    rj = reps[sj]
                    xj = _edges(len(rj), scale, j * scale)

                    for a, ca in enumerate(ri):
                        for b, cb in enumerate(rj):
                            v = rep_corr.loc[ca, cb]
                            img[yi[a]:yi[a+1], xj[b]:xj[b+1]] = v

        # Fill diagonal blocks
        yy, xx = np.ogrid[:scale, :scale]
        lower_left_mask = (yy >= xx)

        for i, si in enumerate(kept):
            ri = reps[si]
            edges = _edges(len(ri), scale)

            block = np.ones((scale, scale), dtype=float)

            for a, ca in enumerate(ri):
                for b, cb in enumerate(ri):
                    v = rep_corr.loc[ca, cb]
                    block[edges[a]:edges[a+1], edges[b]:edges[b+1]] = v

            block[lower_left_mask] = 1.0
            img[i*scale:(i+1)*scale, i*scale:(i+1)*scale] = block

        # ---- FIX: use pcolormesh for exact grid alignment ----
        x_edges = np.arange(n * scale + 1) - 0.5
        y_edges = np.arange(n * scale + 1) - 0.5

        im = ax.pcolormesh(
            x_edges,
            y_edges,
            img,
            cmap=cmap,
            vmin=0,
            vmax=1,
            shading="flat",
        )

        # Keep original orientation
        ax.set_xlim(-0.5, n*scale - 0.5)
        ax.set_ylim(n*scale - 0.5, -0.5)
        ax.set_aspect("auto")

        # Draw main grid
        for k in range(n + 1):
            p = k * scale - 0.5
            ax.axhline(p, color=diag_color, linewidth=0.9)
            ax.axvline(p, color=diag_color, linewidth=0.9)

        # Draw replicate grid
        for i, si in enumerate(kept):
            ri = reps[si]
            yi = _edges(len(ri), scale, i * scale)

            for j, sj in enumerate(kept):
                if i < j:
                    rj = reps[sj]
                    xj = _edges(len(rj), scale, j * scale)

                    for y in yi[1:-1]:
                        ax.plot([j*scale - 0.5, (j+1)*scale - 0.5], [y - 0.5, y - 0.5], color="black", linewidth=0.4)

                    for x in xj[1:-1]:
                        ax.plot([x - 0.5, x - 0.5], [i*scale - 0.5, (i+1)*scale - 0.5], color="black", linewidth=0.4)

                elif i == j:
                    xL, yT = i*scale - 0.5, i*scale - 0.5

                    for y in yi[1:-1]:
                        y = y - 0.5
                        ax.plot([xL + (y - yT), (i+1)*scale - 0.5], [y, y], color="black", linewidth=0.4)

                    for x in yi[1:-1]:
                        x = x - 0.5
                        ax.plot([x, x], [yT, yT + (x - xL)], color="black", linewidth=0.4)

        # Draw diagonal and border
        ax.plot([-0.5, n*scale - 0.5], [-0.5, n*scale - 0.5], color=diag_color, linewidth=diag_lw)
        ax.add_patch(
            plt.Rectangle((0, 0), n*scale, n*scale, fill=False,
                          edgecolor=diag_color, linewidth=diag_lw)
        )

        # Set tick labels
        centers = np.arange(n) * scale + scale / 2 - 0.5
        ax.set_xticks(centers)
        ax.set_yticks(centers)
        ax.set_xticklabels(["ERF" if s == "ERF1" else s for s in kept], rotation=90, fontsize=label_fontsize)
        ax.set_yticklabels(["ERF" if s == "ERF1" else s for s in kept], fontsize=label_fontsize)

        # Clean axis
        ax.set_xlim(-0.5, n*scale - 0.5)
        ax.set_ylim(n*scale - 0.5, -0.5)
        ax.tick_params(length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)

        # Add colorbar
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=12)

        plt.show()

In [ ]:
plot_families_unstitched(dbd_fam_dict, sumprom_filtered, df_dict_representative, label_fontsize=16)